In [11]:
import glfw
from OpenGL.GL import *
import OpenGL.GL.shaders
import numpy as np
import glm


In [12]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE);
window = glfw.create_window(720, 600, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)

In [13]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

In [14]:
fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

In [15]:
# Request a program and shader slots from GPU
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

# Set shaders source
glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

# Compile shaders
glCompileShader(vertex)

if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

# Attach shader objects to the program
glAttachShader(program, vertex)
glAttachShader(program, fragment)

# Build program
glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')
    
# Make program the default program
glUseProgram(program)

# Request a buffer slot from GPU
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)




In [16]:
# preparando espaço para 3 vértices usando 2 coordenadas (x,y)
vertices = np.zeros(0, [("position", np.float32, 3)])

def makecircle(num_vertices, radius):
    circvertices = np.zeros(num_vertices, [("position", np.float32, 3)])

    angle = 0.0
    for counter in range(num_vertices):
        angle += 2*np.pi/num_vertices 
        x = np.cos(angle)*radius
        y = np.sin(angle)*radius
        circvertices[counter]['position'] = [x,y,0.0]
    return circvertices

def Make_sol():
    global solStart, solLen, vertices
    solStart = len(vertices['position'])
    solVertices = makecircle(20, 0.2)
    solLen = len(solVertices['position'])
    vertices = np.concatenate((vertices, solVertices))

def Make_house():
    global houseStart, houseLen, vertices
    houseStart = len(vertices['position'])
    houseVertices = np.zeros(4, [("position", np.float32, 3)])
    houseVertices['position'] = [[0.0,0.0,0.0],
                                 [1.0,0.0,0.0],
                                 [1.0,1.0,0.0],
                                 [0.0,1.0,0.0]]

    houseLen = len(houseVertices['position'])
    vertices = np.concatenate((vertices, houseVertices))
    

Make_sol()
Make_house()
vertices

array([([ 1.9021130e-01,  6.1803401e-02,  0.0000000e+00],),
       ([ 1.6180339e-01,  1.1755705e-01,  0.0000000e+00],),
       ([ 1.1755705e-01,  1.6180339e-01,  0.0000000e+00],),
       ([ 6.1803401e-02,  1.9021130e-01,  0.0000000e+00],),
       ([ 1.2246468e-17,  2.0000000e-01,  0.0000000e+00],),
       ([-6.1803401e-02,  1.9021130e-01,  0.0000000e+00],),
       ([-1.1755705e-01,  1.6180339e-01,  0.0000000e+00],),
       ([-1.6180339e-01,  1.1755705e-01,  0.0000000e+00],),
       ([-1.9021130e-01,  6.1803401e-02,  0.0000000e+00],),
       ([-2.0000000e-01,  2.4492935e-17,  0.0000000e+00],),
       ([-1.9021130e-01, -6.1803401e-02,  0.0000000e+00],),
       ([-1.6180339e-01, -1.1755705e-01,  0.0000000e+00],),
       ([-1.1755705e-01, -1.6180339e-01,  0.0000000e+00],),
       ([-6.1803401e-02, -1.9021130e-01,  0.0000000e+00],),
       ([-3.6739406e-17, -2.0000000e-01,  0.0000000e+00],),
       ([ 6.1803401e-02, -1.9021130e-01,  0.0000000e+00],),
       ([ 1.1755705e-01, -1.6180339e-01,

In [ ]:
def drawSol():

    mat_rotation       = np.array([    [np.cos(angle), -np.sin(angle), 0.0, 0.0], 
                                    [np.sin(angle), np.cos(angle), 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, 0.0], 
                                    [0.0, 1.0, 0.0, 0.8], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_rotation @ mat_translation
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 1.0, 1.0, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLE_FAN, solStart, solLen)



def drawHouse():

    mat_scale      = np.array([    [0.2, 0.0, 0.0, 0.0], 
                                    [0.0, 0.2, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.5], 
                                    [0.0, 1.0, 0.0, 0.01], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_scale @ mat_translation
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.95, 0.95, 0.90, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLE_FAN, houseStart, houseLen)


def drawShadow():

    mat_scale      = np.array([    [-0.2, 0.0, 0.0, 0.0], 
                                    [0.0, -0.2, 0.0, 0.0], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_shearing = np.array([    [1.0, -sc, 0.0, 0.0], 
                                    [0.0, 1.0, 0.0, 0.00], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_translation = np.array([    [1.0, 0.0, 0.0, -0.5], 
                                    [0.0, 1.0, 0.0, 0.01], 
                                    [0.0, 0.0, 1.0, 0.0], 
                                    [0.0, 0.0, 0.0, 1.0]], np.float32)
    
    mat_final = mat_translation @ mat_shearing
    mat_final = mat_scale @ mat_final
    
    loc = glGetUniformLocation(program, "mat_transformation")
    glUniformMatrix4fv(loc, 1, GL_TRUE, mat_final)

  
    glUniform4f(loc_color, 0.0, 0.0, 0.0, 1.0) ### modificando a cor do objeto!
    glDrawArrays(GL_TRIANGLE_FAN, houseStart, houseLen)

In [18]:
glBufferData(GL_ARRAY_BUFFER, vertices['position'].nbytes, vertices['position'], GL_DYNAMIC_DRAW)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)

# Bind the position attribute
# --------------------------------------
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)


loc = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc)
glVertexAttribPointer(loc, 3, GL_FLOAT, False, stride, offset)

loc_color = glGetUniformLocation(program, "color")

R = 0.7
G = 0.0
B = 0.2



In [19]:
# exemplo para matriz de escala

def key_event(window,key,scancode,action,mods):
    global angle, sc
    
   
    if key == 263:
         angle += 0.01 
         sc +=0.02
    if key == 262:
         angle -= 0.01
         sc -=0.02
    
glfw.set_key_callback(window,key_event)


In [20]:
glfw.show_window(window)
angle = 0
sc = 0
while not glfw.window_should_close(window):


    glClear(GL_COLOR_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)
     
    
    drawSol()
    drawHouse()
    drawShadow()
    
    glfw.swap_buffers(window)
    glfw.poll_events() 

glfw.terminate()